# Burgers data generation

Generate the supplied independent smooth-IC Burgers dataset: ten initial conditions, viscosity 0.005, spatial grid 256×256, and 1,200 frames at `dt=0.001`. Temporal downsampling is 4× and FFT spatial downsampling is 8×. The MATLAB key `uv` has HR shape `[2, 1200, 256, 256]` and LR shape `[2, 300, 32, 32]`.

Run the configuration and generation cells. The generator writes `data/burgers/2DBurgers_IC{ic}_ri100_sig040_indIC_...mat`; figures go under `results/data_generation/burgers/figures/`. Set `SLD_DATA_ROOT` and `SLD_RESULTS_ROOT` to override the root directories. Ten HR float32 arrays occupy about 5.86 GiB before metadata, and generation requires additional working memory.

These names match Task 2. Task 1 originally used `ri100` filenames without `sig040_indIC`; its portable notebook warns when it selects this supplied generator dataset. This generator follows manuscript §5.2.3; the original legacy input files were not supplied for comparison. All generation parameters and seeds are unchanged.


In [ ]:
from pathlib import Path
import os

# Find the repository root from Jupyter's current working directory.
_candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((p for p in _candidates
             if (p / "README.md").is_file() and (p / "data_generation").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from the SLD repository or one of its subdirectories.")

DATA_ROOT = Path(os.environ.get("SLD_DATA_ROOT", str(ROOT / "data"))).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get("SLD_RESULTS_ROOT", str(ROOT / "results"))).expanduser().resolve()
base_dir = ROOT
data_dir = DATA_ROOT / "burgers"
result_dir = RESULTS_ROOT / "data_generation" / "burgers"
model_dir = result_dir / "checkpoints"
fig_dir = result_dir / "figures"
model_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {data_dir}")
print(f"Results directory: {result_dir}")

# The generator writes the common dataset consumed by both Burgers notebooks.
data_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 2D Burgers Dataset Generation
# Independent IC + Smooth Perturbation
# HR: [2, 1200, 256, 256]
# LR: [2, 300, 32, 32]
# ============================================================

import os
from pathlib import Path

import numpy as np
import scipy.io as scio
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter




# ============================================================
# Spectral utilities
# ============================================================
def make_wavenumbers(N, L):
    """
    Periodic domain [0, L]^2
    """
    k1d = 2.0 * np.pi * np.fft.fftfreq(N, d=L / N)
    kx, ky = np.meshgrid(k1d, k1d, indexing="ij")
    k2 = kx**2 + ky**2
    return kx, ky, k2


def dealias_mask(N):
    """
    2/3 dealiasing rule mask.
    """
    cutoff = N // 3
    freq = np.fft.fftfreq(N) * N
    fx, fy = np.meshgrid(freq, freq, indexing="ij")
    mask = (np.abs(fx) < cutoff) & (np.abs(fy) < cutoff)
    return mask.astype(np.float64)


def fft2(f):
    return np.fft.fft2(f)


def ifft2(fh):
    return np.fft.ifft2(fh).real


# ============================================================
# Initial condition utilities
# ============================================================
def scale_to_range(a, lo=0.1, hi=1.1):
    """
    Scale array to [lo, hi].
    """
    amin = a.min()
    amax = a.max()

    if amax - amin < 1e-12:
        return np.full_like(a, 0.5 * (lo + hi))

    return lo + (a - amin) * (hi - lo) / (amax - amin)


def normalize_zero_mean_unit_std(a):
    """
    Normalize field to zero mean and unit standard deviation.
    """
    a = a - a.mean()
    a = a / (a.std() + 1e-12)
    return a


def generate_fourier_base_initial_condition(
    N,
    L,
    base_seed=1,
    mode_min=-4,
    mode_max=4,
    output_range=(0.1, 1.1),
):
    """
    Generate one Fourier random base IC.

    Different base_seed gives different large-scale structures.

    u0 = sum r_ij^(1) sin(ix+jy) + r_ij^(2) cos(ix+jy)
    v0 = sum r_ij^(3) sin(ix+jy) + r_ij^(4) cos(ix+jy)
    """
    rng = np.random.default_rng(base_seed)

    x = np.linspace(-L / 2, L / 2, N, endpoint=False)
    y = np.linspace(-L / 2, L / 2, N, endpoint=False)
    X, Y = np.meshgrid(x, y, indexing="ij")

    u0 = np.zeros((N, N), dtype=np.float64)
    v0 = np.zeros((N, N), dtype=np.float64)

    for i in range(mode_min, mode_max + 1):
        for j in range(mode_min, mode_max + 1):
            # 跳过全零模式，避免引入不必要常数项
            if i == 0 and j == 0:
                continue

            r1 = rng.standard_normal()
            r2 = rng.standard_normal()
            r3 = rng.standard_normal()
            r4 = rng.standard_normal()

            phase = 2.0 * np.pi * (i * X / L + j * Y / L)

            u0 += r1 * np.sin(phase) + r2 * np.cos(phase)
            v0 += r3 * np.sin(phase) + r4 * np.cos(phase)

    lo, hi = output_range
    u0 = scale_to_range(u0, lo, hi)
    v0 = scale_to_range(v0, lo, hi)

    return u0.astype(np.float32), v0.astype(np.float32)


def generate_smooth_noise(
    N,
    rng,
    sigma=4.0,
    amplitude=0.1,
):
    """
    Generate smooth perturbation noise.

    Steps:
    1. sample white noise
    2. Gaussian smooth
    3. normalize to zero mean / unit std
    4. multiply by amplitude
    """
    noise = rng.standard_normal((N, N)).astype(np.float32)

    # 平滑噪声，避免像素级高频雪花
    noise = gaussian_filter(noise, sigma=sigma)

    # 归一化后再乘 amplitude，这样不同 IC 的扰动强度一致
    noise = normalize_zero_mean_unit_std(noise)

    noise = amplitude * noise

    return noise.astype(np.float32)


def get_initial_u_and_v_burgers(
    N,
    L,
    ic_seed=1,
    base_seed=1,
    random_influence=0.1,
    smooth_noise_sigma=4.0,
    mode_min=-4,
    mode_max=4,
    output_range=(0.1, 1.1),
    clip_range=None,
):
    """
    Independent Burgers IC.

    Important:
    - each IC uses its own base_seed
    - each IC also uses its own smooth perturbation seed
    - no shared mother IC

    u0 = u_base_ic + random_influence * smooth_noise_u
    v0 = v_base_ic + random_influence * smooth_noise_v
    """

    # 1. 每个 IC 生成不同的大尺度 Fourier base field
    u_base, v_base = generate_fourier_base_initial_condition(
        N=N,
        L=L,
        base_seed=base_seed,
        mode_min=mode_min,
        mode_max=mode_max,
        output_range=output_range,
    )

    u = u_base.copy()
    v = v_base.copy()

    # 2. 每个 IC 加不同的平滑扰动
    if random_influence > 0:
        rng_noise = np.random.default_rng(ic_seed)

        noise_u = generate_smooth_noise(
            N=N,
            rng=rng_noise,
            sigma=smooth_noise_sigma,
            amplitude=random_influence,
        )

        noise_v = generate_smooth_noise(
            N=N,
            rng=rng_noise,
            sigma=smooth_noise_sigma,
            amplitude=random_influence,
        )

        u += noise_u
        v += noise_v

    # 3. 可选 clip，避免极端初值导致数值不稳定
    if clip_range is not None:
        lo, hi = clip_range
        u = np.clip(u, lo, hi)
        v = np.clip(v, lo, hi)

    return u.astype(np.float32), v.astype(np.float32)


# ============================================================
# Burgers RHS
# ============================================================
def spectral_derivatives(f, kx, ky):
    fh = fft2(f)
    fx = ifft2(1j * kx * fh)
    fy = ifft2(1j * ky * fh)
    return fx, fy


def laplacian(f, k2):
    fh = fft2(f)
    return ifft2(-k2 * fh)


def rhs_burgers(u, v, nu, kx, ky, k2, mask=None):
    """
    2D vector Burgers:

        u_t + u u_x + v u_y = nu Δu
        v_t + u v_x + v v_y = nu Δv
    """
    ux, uy = spectral_derivatives(u, kx, ky)
    vx, vy = spectral_derivatives(v, kx, ky)

    adv_u = u * ux + v * uy
    adv_v = u * vx + v * vy

    if mask is not None:
        adv_u = ifft2(fft2(adv_u) * mask)
        adv_v = ifft2(fft2(adv_v) * mask)

    du = -adv_u + nu * laplacian(u, k2)
    dv = -adv_v + nu * laplacian(v, k2)

    return du, dv


def rk4_step_burgers(u, v, dt, nu, kx, ky, k2, mask=None):
    k1u, k1v = rhs_burgers(u, v, nu, kx, ky, k2, mask=mask)

    k2u, k2v = rhs_burgers(
        u + 0.5 * dt * k1u,
        v + 0.5 * dt * k1v,
        nu,
        kx,
        ky,
        k2,
        mask=mask,
    )

    k3u, k3v = rhs_burgers(
        u + 0.5 * dt * k2u,
        v + 0.5 * dt * k2v,
        nu,
        kx,
        ky,
        k2,
        mask=mask,
    )

    k4u, k4v = rhs_burgers(
        u + dt * k3u,
        v + dt * k3v,
        nu,
        kx,
        ky,
        k2,
        mask=mask,
    )

    u_next = u + (dt / 6.0) * (k1u + 2 * k2u + 2 * k3u + k4u)
    v_next = v + (dt / 6.0) * (k1v + 2 * k2v + 2 * k3v + k4v)

    return u_next, v_next


# ============================================================
# Visualization
# ============================================================
def postProcess(output, N, xmin, xmax, ymin, ymax, num, batch, save_dir):
    """
    output: [2, T, H, W] or [T, 2, H, W]
    """
    x = np.linspace(xmin, xmax, N, endpoint=False)
    y = np.linspace(ymin, ymax, N, endpoint=False)
    x_star, y_star = np.meshgrid(x, y, indexing="ij")

    if output.shape[0] == 2:
        u_pred = output[0, num, :, :]
        v_pred = output[1, num, :, :]
    else:
        u_pred = output[num, 0, :, :]
        v_pred = output[num, 1, :, :]

    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(6, 3))
    fig.subplots_adjust(hspace=0.3, wspace=0.3)

    cf = ax[0].scatter(
        x_star,
        y_star,
        c=u_pred,
        alpha=0.95,
        edgecolors="none",
        cmap="RdBu_r",
        marker="s",
        s=2,
    )
    ax[0].axis("square")
    ax[0].set_xlim([xmin, xmax])
    ax[0].set_ylim([ymin, ymax])
    ax[0].set_title("u")
    fig.colorbar(cf, ax=ax[0])

    cf = ax[1].scatter(
        x_star,
        y_star,
        c=v_pred,
        alpha=0.95,
        edgecolors="none",
        cmap="RdBu_r",
        marker="s",
        s=2,
    )
    ax[1].axis("square")
    ax[1].set_xlim([xmin, xmax])
    ax[1].set_ylim([ymin, ymax])
    ax[1].set_title("v")
    fig.colorbar(cf, ax=ax[1])

    save_path = save_dir / f"uv_[IC={batch}][t={num}].png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close("all")


# ============================================================
# Downsampling utilities
# ============================================================
def fft_crop2d_centered(F, out_h, out_w):
    H, W = F.shape

    if out_h > H or out_w > W:
        raise ValueError("out size must not exceed input size")

    Fs = np.fft.fftshift(F)

    cy, cx = H // 2, W // 2
    hy, hx = out_h // 2, out_w // 2

    if out_h % 2 == 0:
        ys = cy - hy
        ye = cy + hy
    else:
        ys = cy - hy
        ye = cy + hy + 1

    if out_w % 2 == 0:
        xs = cx - hx
        xe = cx + hx
    else:
        xs = cx - hx
        xe = cx + hx + 1

    Fc = Fs[ys:ye, xs:xe]

    return np.fft.ifftshift(Fc)


def downsample_fft_single(field2d, out_size):
    H, W = field2d.shape

    if H != W:
        raise ValueError(f"Only square input supported, got {field2d.shape}")

    if H % out_size != 0:
        raise ValueError(f"Input size {H} must be divisible by out_size {out_size}")

    F = np.fft.fft2(field2d)
    Fc = fft_crop2d_centered(F, out_size, out_size)

    lr = np.fft.ifft2(Fc).real

    # amplitude correction after Fourier cropping
    lr *= (out_size * out_size) / (H * W)

    return lr.astype(field2d.dtype, copy=False)


def downsample_spatial_fft(data_high, n_levels=3):
    """
    data_high: [2, T, H, W]
    output:    [2, T, H/2^n_levels, W/2^n_levels]
    """
    data_high = np.asarray(data_high)

    C, T, H, W = data_high.shape

    if H != W:
        raise ValueError(f"Only square input supported, got {H}x{W}")

    out_size = H // (2 ** n_levels)

    out = np.zeros((C, T, out_size, out_size), dtype=data_high.dtype)

    for c in range(C):
        for t in range(T):
            out[c, t] = downsample_fft_single(data_high[c, t], out_size)

    return out


def downsample_temporal(data, t_ds=4):
    """
    data: [2, T, H, W]
    """
    return data[:, ::t_ds, :, :]


# ============================================================
# Dataset generation
# ============================================================
def generate_dataset(
    ic_number,
    nu=0.005,
    N=256,
    L=2.0,
    dt=1.0e-3,
    T_hr_target=1200,
    random_influence=0.08,
    smooth_noise_sigma=4.0,
    temporal_downsample_factor=4,
    spatial_downsample_levels=3,
    base_ic_seed=2027,
    mode_min=-4,
    mode_max=4,
    output_range=(0.1, 1.1),
    clip_range=(-0.2, 1.4),
):
    """
    Independent smooth-IC 2D Burgers dataset generation.

    - each IC has different Fourier base field
    - each IC has different smooth perturbation
    - HR: [2, T, 256, 256]
    - LR: [2, T/4, 32, 32]
    """

    x_min, x_max = -L / 2, L / 2
    y_min, y_max = -L / 2, L / 2
    dx = L / N

    save_every = 1
    total_steps = T_hr_target - 1
    actual_T = total_steps * dt

    TEMPORAL_DS = temporal_downsample_factor
    SPATIAL_DS = 2 ** spatial_downsample_levels

    T_lr_target = (T_hr_target + TEMPORAL_DS - 1) // TEMPORAL_DS
    H_lr = N // SPATIAL_DS
    W_lr = N // SPATIAL_DS

    ri_tag = f"ri{int(random_influence * 1000):03d}"
    sig_tag = f"sig{int(smooth_noise_sigma * 10):03d}"
    seed_tag = f"seed{base_ic_seed}"

    print(f"\n{'=' * 60}")
    print(f"Generating 2D Burgers dataset for IC={ic_number}")
    print(f"{'=' * 60}")
    print(f"HR resolution: {N}x{N}")
    print(f"Temporal downsampling: {TEMPORAL_DS}x")
    print(f"Spatial downsampling: {SPATIAL_DS}x")
    print(f"dt: {dt}, total simulation steps: {total_steps}, actual_T: {actual_T:.6f}")
    print(f"nu: {nu}, dx: {dx:.6e}, domain: [{x_min}, {x_max}) x [{y_min}, {y_max})")
    print(f"base_ic_seed: {base_ic_seed}")
    print(f"random_influence: {random_influence}")
    print(f"smooth_noise_sigma: {smooth_noise_sigma}")
    print(f"Fourier modes: [{mode_min}, {mode_max}]")
    print(f"output_range before perturbation: {output_range}")
    print(f"clip_range after perturbation: {clip_range}")

    kx, ky, k2 = make_wavenumbers(N, L)
    mask = dealias_mask(N)

    # ========================================================
    # Generate independent smooth initial condition
    # ========================================================
    u, v = get_initial_u_and_v_burgers(
        N=N,
        L=L,
        ic_seed=10_000 + ic_number,
        base_seed=base_ic_seed,
        random_influence=random_influence,
        smooth_noise_sigma=smooth_noise_sigma,
        mode_min=mode_min,
        mode_max=mode_max,
        output_range=output_range,
        clip_range=clip_range,
    )

    print(f"Initial u range: [{u.min():.4e}, {u.max():.4e}], mean={u.mean():.4e}, std={u.std():.4e}")
    print(f"Initial v range: [{v.min():.4e}, {v.max():.4e}], mean={v.mean():.4e}, std={v.std():.4e}")

    u_record = u.copy()[None, ...]
    v_record = v.copy()[None, ...]

    # ========================================================
    # Time integration
    # ========================================================
    for step in range(total_steps):
        u, v = rk4_step_burgers(
            u=u,
            v=v,
            dt=dt,
            nu=nu,
            kx=kx,
            ky=ky,
            k2=k2,
            mask=mask,
        )

        # Safety check
        if np.isnan(u).any() or np.isnan(v).any() or np.isinf(u).any() or np.isinf(v).any():
            raise FloatingPointError(f"NaN/Inf detected at IC={ic_number}, step={step + 1}")

        if (step + 1) % save_every == 0:
            if step % 200 == 0:
                print(f"  Progress: {step + 1}/{total_steps}")

            u_record = np.concatenate((u_record, u[None, ...]), axis=0)
            v_record = np.concatenate((v_record, v[None, ...]), axis=0)

    print(f"  Simulation complete: {u_record.shape[0]} timesteps")

    UV_high = np.stack([u_record, v_record], axis=0).astype(np.float32)

    # ========================================================
    # Save HR
    # ========================================================
    save_path_high = data_dir / (
        f"2DBurgers_IC{ic_number}_{ri_tag}_{sig_tag}_indIC_"
        f"2x{T_hr_target}x{N}x{N}.mat"
    )

    scio.savemat(
        save_path_high,
        {
            "uv": UV_high,
            "ic_number": ic_number,
            "nu": nu,
            "N": N,
            "L": L,
            "dt": dt,
            "random_influence": random_influence,
            "smooth_noise_sigma": smooth_noise_sigma,
            "base_ic_seed": base_ic_seed,
            "mode_min": mode_min,
            "mode_max": mode_max,
        },
    )

    print(f"  ✓ Saved HR: {save_path_high.name} | shape: {UV_high.shape}")

    # ========================================================
    # Downsample
    # ========================================================
    UV_temp = downsample_temporal(UV_high, t_ds=TEMPORAL_DS)
    print(f"  After temporal DS: {UV_temp.shape}")

    UV_low = downsample_spatial_fft(UV_temp, n_levels=spatial_downsample_levels)
    print(f"  After spatial DS: {UV_low.shape}")

    # ========================================================
    # Save LR
    # ========================================================
    save_path_low = data_dir / (
        f"2DBurgers_IC{ic_number}_{ri_tag}_{sig_tag}_indIC_"
        f"2x{UV_low.shape[1]}x{H_lr}x{W_lr}.mat"
    )

    scio.savemat(
        save_path_low,
        {
            "uv": UV_low,
            "ic_number": ic_number,
            "nu": nu,
            "N_hr": N,
            "H_lr": H_lr,
            "W_lr": W_lr,
            "L": L,
            "dt_hr": dt,
            "temporal_downsample_factor": TEMPORAL_DS,
            "spatial_downsample_factor": SPATIAL_DS,
            "random_influence": random_influence,
            "smooth_noise_sigma": smooth_noise_sigma,
            "base_ic_seed": base_ic_seed,
            "mode_min": mode_min,
            "mode_max": mode_max,
        },
    )

    print(f"  ✓ Saved LR: {save_path_low.name} | shape: {UV_low.shape}")

    # ========================================================
    # Verification
    # ========================================================
    assert UV_high.shape == (2, T_hr_target, N, N), f"HR shape mismatch: {UV_high.shape}"
    assert UV_low.shape == (2, T_lr_target, H_lr, W_lr), f"LR shape mismatch: {UV_low.shape}"

    print("  ✓ Shape verification passed")

    # ========================================================
    # Optional visualization
    # ========================================================
    if ic_number in [1, 5, 10]:
        print("  Generating visualizations...")
        for t_idx in [0, 200, 400, 600, 800, 1000]:
            if t_idx < UV_high.shape[1]:
                postProcess(
                    output=UV_high,
                    N=N,
                    xmin=x_min,
                    xmax=x_max,
                    ymin=y_min,
                    ymax=y_max,
                    num=t_idx,
                    batch=ic_number,
                    save_dir=fig_dir,
                )

    return UV_high, UV_low


# ============================================================
# Main execution
# ============================================================
if __name__ == "__main__":
    print("=" * 60)
    print("2D Burgers Dataset Generation")
    print("Independent IC + Smooth Perturbation")
    print("SR x8, Temporal x4")
    print("=" * 60)
    print(f"Data directory: {data_dir.absolute()}")
    print(f"Figure directory: {fig_dir.absolute()}")

    # ========================================================
    # Main parameters
    # ========================================================
    N_ICS = 10

    BASE_SEED = 2026

    # 平滑扰动强度
    # 建议先用 0.08 或 0.10
    RANDOM_INFLUENCE = 0.1

    # 平滑噪声 sigma
    # 越大越平滑；推荐 3.0 到 6.0
    SMOOTH_NOISE_SIGMA = 4.0

    for ic in range(1, N_ICS + 1):
        generate_dataset(
            ic_number=ic,
            nu=0.005,
            N=256,
            L=2.0,
            dt=1e-3,
            T_hr_target=1200,
            random_influence=RANDOM_INFLUENCE,
            smooth_noise_sigma=SMOOTH_NOISE_SIGMA,
            temporal_downsample_factor=4,
            spatial_downsample_levels=3,
            base_ic_seed=BASE_SEED + ic,
            mode_min=-4,
            mode_max=4,
            output_range=(0.1, 1.1),
            clip_range=(-0.2, 1.4),
        )

    print("\n" + "=" * 60)
    print("2D Burgers Dataset Generation Complete!")
    print(f"Created {len(list(data_dir.glob('*_2x1200x256x256.mat')))} HR files")
    print(f"Created {len(list(data_dir.glob('*_2x300x32x32.mat')))} LR files")
    print("=" * 60)


## Optional initial-condition check

Run this cell after generating ICs 1, 2, and 3. It loads their HR files and prints pairwise differences between their initial conditions; it does not require a trained checkpoint.


In [ ]:
import os
import numpy as np
import scipy.io as sio

ic_ids = [1, 2, 3]

def find_hr_file(base_dir, ic_id):
    target_name = f"2DBurgers_IC{ic_id}_ri100_sig040_indIC_2x1200x256x256.mat"

    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f == target_name:
                return os.path.join(root, f)

    raise FileNotFoundError(f"Cannot find {target_name}")


def load_4d_mat(path):
    mat = sio.loadmat(path)

    for k, v in mat.items():
        if isinstance(v, np.ndarray) and v.ndim == 4:
            print(f"Loaded {os.path.basename(path)} | key={k} | shape={v.shape}")
            return v

    raise ValueError(f"No 4D array found in {path}")


ics = {}

for ic_id in ic_ids:
    path = find_hr_file(data_dir, ic_id)
    data = load_4d_mat(path)
    ics[ic_id] = data[:, 0]   # shape: [2, H, W], initial condition


# pairwise difference
for i in range(len(ic_ids)):
    for j in range(i + 1, len(ic_ids)):
        a = ics[ic_ids[i]]
        b = ics[ic_ids[j]]

        max_abs_diff = np.max(np.abs(a - b))
        mean_abs_diff = np.mean(np.abs(a - b))
        rel_diff = np.linalg.norm(a - b) / (np.linalg.norm(a) + 1e-12)

        print(f"IC{ic_ids[i]} vs IC{ic_ids[j]}:")
        print(f"  max_abs_diff  = {max_abs_diff:.6e}")
        print(f"  mean_abs_diff = {mean_abs_diff:.6e}")
        print(f"  rel_diff      = {rel_diff:.6e}")
